# GCN: Spatial Gene Imputation Post-processing (Pseudo-6K Benchmark)

The GCN model (Graph Convolutional Network) is trained and run using `imputation_model/Tutorial.ipynb`.
This notebook handles the benchmark-specific post-processing step: loading the raw GCN predictions
for the pseudo-6K held-out test set, merging with the observed 6K counts, and saving the combined
CSV for downstream comparison in `01_imputation_comparison.ipynb`.

**Pseudo-6K setup**: WTX cells are split by FOV; the test split is presented to the model with only
6K genes observed; the model imputes the remaining ~12K.

**Input**: Raw GCN prediction CSV (`Gene_imputation_best_log1p.csv` from Tutorial.ipynb, run on pseudo-6K)  
**Output**: `half_ori_half_ywlb_imp_log1p.csv` — combined 6K observed + GCN-imputed 12K genes


In [ ]:
import pandas as pd
import scanpy as sc
import numpy as np

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR  = "/path/to/cosmx/data"
GCN_DIR   = "/path/to/gcn/results"     # output directory from Tutorial.ipynb pseudo-6K run
OUT_DIR   = "/path/to/cosmx/count_csv"

In [ ]:
# Load WTX data to define test cells
adata = sc.read_h5ad(f"{DATA_DIR}/qc_wtx_v7_UC_P1_UC_P2_anno.h5ad")
adata.obs['fov'] = adata.obs['fov'].astype(int)

# Train/test split by FOV (must match the split used during GCN inference)
train_cells, test_cells = [], []
for fov, cell in zip(adata.obs['fov'], adata.obs['fov_cell_id']):
    if (fov > 9 and fov <= 32) or (fov > 44):
        train_cells.append(cell)
    else:
        test_cells.append(cell)
print(f"Train: {len(train_cells):,}  |  Test: {len(test_cells):,}")

adata_test = adata[adata.obs['fov_cell_id'].isin(test_cells), :].copy()

# 6K gene panel
genes_6k = pd.read_csv(f"{DATA_DIR}/6k_actual_gene_names.csv")['x'].tolist()

In [ ]:
# Load GCN raw predictions (best-epoch model, output of Tutorial.ipynb pseudo-6K run)
# 'b' = best epoch (used in paper as GCN); 'f' = final epoch
gcn_pred = pd.read_csv(f"{GCN_DIR}/Gene_imputation_best_log1p.csv", index_col=0)

# Filter to test cells
gcn_pred_test = gcn_pred[gcn_pred.index.isin(test_cells)].copy()
print(f"GCN predictions: {gcn_pred_test.shape[0]:,} test cells x {gcn_pred_test.shape[1]:,} genes")

In [ ]:
# Observed 6K counts (ground truth panel genes for test cells)
raw_6k = pd.DataFrame(adata_test[:, adata_test.var.index.isin(genes_6k)].X)
raw_6k.columns = [g for g in adata_test.var_names if g in genes_6k]
raw_6k.index   = adata_test.obs['fov_cell_id'].tolist()

# Combine: keep 6K observed counts; use GCN for imputed 12K genes
imputed_genes = [g for g in gcn_pred_test.columns if g not in genes_6k]
combined = pd.concat([raw_6k, gcn_pred_test[imputed_genes]], axis=1)

combined.to_csv(f"{OUT_DIR}/half_ori_half_ywlb_imp_log1p.csv")
print(f"Exported: {combined.shape[0]:,} cells x {combined.shape[1]:,} genes")